# 01 · Run one question, end to end

This notebook runs **one** question from the clinical-retrieval benchmark
through the harness, end to end, and names every setup trap along the way —
the same traps documented in `04-benchmarks/clinical-retrieval/README.md` and
this stage's `README.md`, shown here in the place they actually bite.

**Honesty check first:** the retrieval pipeline this benchmark measures is
flaky, not consistently broken — on a run of six against the identical
question and configuration, it retrieved real papers five times and none
once. This notebook's default path — no backend configured — always shows
the zero-papers state regardless, since nothing is wired up to retrieve
from; that default is a property of running with no backend, not a claim
about the real pipeline's typical run. Point it at a real backend (see the
last section) to see what an actual run does.

## What this notebook demonstrates

| Name | What it does | One example |
|---|---|---|
| Load a benchmark question | Pulls one real question record by id from `questions.py` | `case = BENCHMARK_QUERIES` entry with `id == "B01"` |
| Domain-routing test defect note | Documents a pre-existing, unrelated unit-test defect before it can be mistaken for something this notebook broke | `KNOWN_DOMAIN_TEST_DEFECT` dict |
| Groq / Cloudflare User-Agent probe | Shows why a raw `urllib` request to `api.groq.com` gets a Cloudflare 403, and why the `openai` SDK doesn't | `probe_groq_with_default_urllib()` |
| Offline `run_one_question` harness | Runs the same `retrieval_fn` / `generation_fn` interface `eval.py` uses, with no backend configured | `run_one_question(case, default_retrieval_fn, default_generation_fn)` |
| Point at a real backend | Shows the env-var wiring to swap in a real retrieval/generation backend | `EVAL_RETRIEVAL_BACKEND=mymodule:my_retrieval_fn python eval.py` |


## Step 1 — locate the repo root and confirm the environment

This notebook lives two levels below the repo root, so the first thing it does is walk up the directory tree to find `nbio.py` and import it — everything else in this notebook depends on `repo_root` being set correctly.

In [ ]:
# This notebook lives two levels below the repo root (01-modules/06-bench/),
# and Jupyter starts a kernel with its working directory set to the
# notebook's own folder -- so nbio.py (at the repo root) is not importable
# yet. Walk up until we find it, same logic nbio.bootstrap() uses
# internally once it CAN be imported.
import sys
from pathlib import Path

def _find_repo_root(start):
    root = start.resolve()
    for _ in range(6):
        if (root / "nbio.py").is_file():
            return root
        root = root.parent
    raise RuntimeError("could not locate nbio.py above the current directory")

_repo_root_for_import = _find_repo_root(Path.cwd())
if str(_repo_root_for_import) not in sys.path:
    sys.path.insert(0, str(_repo_root_for_import))

import nbio
repo_root = nbio.bootstrap()
nbio.show_environment()

## Step 2 — load a real question

The 20 questions live in `04-benchmarks/clinical-retrieval/questions.py`.
They are clinical *topics* to retrieve papers about, not patient data.

In [ ]:
import sys
from pathlib import Path

BENCH_DIR = repo_root / "04-benchmarks" / "clinical-retrieval"
sys.path.insert(0, str(BENCH_DIR))

from questions import BENCHMARK_QUERIES

case = next(c for c in BENCHMARK_QUERIES if c["id"] == "B01")
print(f"id:                {case['id']}")
print(f"query:             {case['query']}")
print(f"expected_domains:  {case['expected_domains']}")
print(f"required_keywords: {case['required_keywords']}")
print(f"min_evidence:      {case['min_evidence_level']}")

## Step 3 — three unit tests fail on any non-default department

Three unit tests fail whenever `DOMAIN` is set to a department other than
`plastic_surgery` — they assert `burn_trauma` routing, which a department
like `breast_reconstruction` does not have.

This is **not relevant to running this notebook** (there's no `DOMAIN`
setting here), but it costs a full day to debug cold if you go on to wire a
real backend and start seeing exactly three failures tied to domain routing.
It's a defect in the tests, not in your install — worth knowing before you go
looking for what you broke.

In [ ]:
# Illustrative only -- this notebook has no DOMAIN setting or department
# config, so there's nothing to run here. This cell documents the shape of
# the trap so it's searchable from the notebook, not just the README.
KNOWN_DOMAIN_TEST_DEFECT = {
    "symptom": "3 of 123 unit tests fail",
    "trigger": "DOMAIN set to anything other than plastic_surgery",
    "cause": "tests assert burn_trauma routing; other departments (e.g. "
             "breast_reconstruction) don't have that route by design",
    "fix": "none needed on your end -- it's a test defect, not an install problem",
}
for k, v in KNOWN_DOMAIN_TEST_DEFECT.items():
    print(f"{k:>10}: {v}")

## Step 4 — Groq's API rejects Python's default User-Agent

`api.groq.com` sits behind Cloudflare, which returns an HTTP 403 (Cloudflare
error 1010) to requests carrying Python's default `urllib`/`requests`
User-Agent string. The `openai` SDK sets its own User-Agent and is
unaffected — this is why the project calls Groq through the `openai` SDK
pointed at Groq's base URL, rather than rolling raw HTTP calls.

The cell below actually attempts both, with a short timeout, and reports
what happened — network access may or may not be available in the
environment running this notebook, so it degrades to reporting *why* rather
than failing the notebook either way.

In [ ]:
import urllib.request
import urllib.error

def probe_groq_with_default_urllib(timeout=5):
    req = urllib.request.Request("https://api.groq.com/openai/v1/models")
    try:
        urllib.request.urlopen(req, timeout=timeout)
        return "unexpected success -- Cloudflare didn't block this request"
    except urllib.error.HTTPError as exc:
        if exc.code == 403:
            return "403 (Cloudflare error 1010) -- exactly the trap: default urllib User-Agent was rejected"
        return f"HTTP error {exc.code} -- not the User-Agent trap, something else"
    except Exception as exc:  # noqa: BLE001 -- no network in this environment is also a valid outcome
        return f"could not reach api.groq.com at all ({exc.__class__.__name__}) -- no network here, trap not observable"


## Step 5 — run the probe and read the result

In [ ]:
print("hand-rolled urllib probe:", probe_groq_with_default_urllib())
print()
print("fix: use the `openai` SDK pointed at Groq's base URL instead --")
print("  from openai import OpenAI")
print('  client = OpenAI(api_key=GROQ_API_KEY, base_url="https://api.groq.com/openai/v1")')
print("  client.chat.completions.create(...)  # sets its own User-Agent, unaffected")

## Step 6 — run the question through the harness (offline path)

This is the same interface `04-benchmarks/clinical-retrieval/eval.py` uses:
a `retrieval_fn(query) -> (papers, error)` and a
`generation_fn(query, papers) -> answer`. With no backend configured, both
default to an honest "nothing configured" result rather than a confident
fabricated answer — deliberately, since it is easy for a RAG pipeline to
retrieve nothing and still answer confidently with specific clinical numbers
anyway.

In [ ]:
import time

def default_retrieval_fn(query):
    # No backend configured: zero papers, no error.
    return [], None

## Step 7 — the generation stand-in, with no backend configured

In [ ]:
def default_generation_fn(query, papers):
    # No backend configured: say so, not a confident guess.
    return "[no generation backend configured]"

## Step 8 — the harness function that ties retrieval and generation together

In [ ]:
def run_one_question(case, retrieval_fn, generation_fn):
    t0 = time.monotonic()
    papers, error = retrieval_fn(case["query"])
    papers = papers or []
    latency_ms = int((time.monotonic() - t0) * 1000)

    all_text = " ".join(
        (p.get("summary") or p.get("text") or p.get("title") or "").lower()
        for p in papers
    )
    required = case.get("required_keywords", [])
    found = [kw for kw in required if kw.lower() in all_text]

    answer = generation_fn(case["query"], papers)

    return {
        "id": case["id"],
        "error": error,
        "paper_count": len(papers),
        "keywords_found": found,
        "keyword_precision": round(len(found) / len(required), 3) if required else 0.0,
        "latency_ms": latency_ms,
        "answer": answer,
    }

## Step 9 — run it and look at the result

In [ ]:
result = run_one_question(case, default_retrieval_fn, default_generation_fn)
for k, v in result.items():
    print(f"{k:>16}: {v}")

## Step 10 — point this at a real backend instead

Replace `default_retrieval_fn` / `default_generation_fn` above with functions
backed by a real pipeline (yours, or one you're porting) and re-run. The
signature is fixed so `eval.py` can run the same functions across all 20
questions:

```python
def my_retrieval_fn(query: str) -> tuple[list[dict], str | None]:
    ...  # return (papers, error) -- papers is a list of {"title", "summary"/"text", ...}

def my_generation_fn(query: str, papers: list[dict]) -> str:
    ...  # return the answer text
```

Then from `04-benchmarks/clinical-retrieval/`:

```bash
EVAL_RETRIEVAL_BACKEND=mymodule:my_retrieval_fn \
EVAL_GENERATION_BACKEND=mymodule:my_generation_fn \
python eval.py
```

## What "report every setup problem hit" (open task 3) looks like

If you do wire up a real backend, `LEADERBOARD.md` task 3 asks for exactly
what this notebook demonstrated for two known traps: a plain list of every
problem you hit, in the order you hit it, each with the one-line fix. That's
the deliverable — not a working pipeline, just an honest list.